In [1]:
# ==========================================================
# Notebook 02 - Geospatial Preprocessing
# Cell 1 - Import Libraries
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

import geopandas as gpd

import rasterio
from rasterio.merge import merge
from rasterio.mask import mask

from rasterio.warp import (
    calculate_default_transform,
    reproject,
    Resampling
)

from rasterio.enums import Resampling as RS

from rasterio.plot import show

from shapely.geometry import mapping

import matplotlib.pyplot as plt

print("="*70)
print("Notebook 02")
print("Geospatial Preprocessing")
print("="*70)

Notebook 02
Geospatial Preprocessing


In [2]:
# ==========================================================
# Cell 2 - Folder Structure
# ==========================================================

from pathlib import Path

# Project Root
PROJECT_DIR = Path.cwd()

# Data Folder
DATA_DIR = PROJECT_DIR / "Data"

BOUNDARY_DIR = DATA_DIR / "Boundaries"
RAINFALL_DIR = DATA_DIR / "Rainfall"
DEM_DIR = DATA_DIR / "DEM"
NDVI_DIR = DATA_DIR / "NDVI"
LANDCOVER_DIR = DATA_DIR / "LandCover"
SOIL_DIR = DATA_DIR / "Soil"
POPULATION_DIR = DATA_DIR / "Population"
LANDSLIDE_DIR = DATA_DIR / "LandslideInventory"
OSM_DIR = DATA_DIR / "OpenStreetMap"

# Output Folder
PROCESSED_DIR = PROJECT_DIR / "Processed"
PROCESSED_DIR.mkdir(exist_ok=True)

print("Project :", PROJECT_DIR)
print("Data    :", DATA_DIR)
print("Processed:", PROCESSED_DIR)

Project : /Users/rajdeepmandal/Desktop/PrithviX
Data    : /Users/rajdeepmandal/Desktop/PrithviX/Data
Processed: /Users/rajdeepmandal/Desktop/PrithviX/Processed


In [3]:
# ==========================================================
# Cell 3 - Verify Input Files
# ==========================================================

folders = {
    "DEM": DEM_DIR,
    "NDVI": NDVI_DIR,
    "LandCover": LANDCOVER_DIR,
    "Soil": SOIL_DIR,
    "Population": POPULATION_DIR,
    "Boundaries": BOUNDARY_DIR,
    "OSM": OSM_DIR,
    "Landslides": LANDSLIDE_DIR
}

print("="*70)

for name, folder in folders.items():

    print(f"\n{name}")

    files = list(folder.glob("*"))

    print("-"*40)

    for f in files:
        print(f.name)

print("\nDone.")


DEM
----------------------------------------
.DS_Store
ner_dem.tif

NDVI
----------------------------------------
ner_ndvi2.tif
.DS_Store
ner_ndvi.tif

LandCover
----------------------------------------
.DS_Store
landcover_tile_4.tif
landcover_tile_1.tif
landcover_tile_3.tif
landcover_tile_2.tif

Soil
----------------------------------------
.DS_Store
soil.tif

Population
----------------------------------------
.DS_Store
south_population.tif
north_population.tif
northeastern_population.tif

Boundaries
----------------------------------------
India_ADM2_GeoBoundaries_2025.geojson

OSM
----------------------------------------
.DS_Store
southern-zone.gpkg
north-eastern-zone.gpkg
northern-zone.gpkg

Landslides
----------------------------------------
.DS_Store
landslides.csv

Done.


In [4]:
# ==========================================================
# Cell 4 - Merge NDVI Tiles
# ==========================================================

from rasterio.merge import merge
import rasterio

# Get all NDVI files
ndvi_files = sorted(NDVI_DIR.glob("*.tif"))

print("=" * 70)
print("NDVI Files Found")
print("=" * 70)

for f in ndvi_files:
    print(f.name)

# Open rasters
src_files = [rasterio.open(f) for f in ndvi_files]

# Merge
ndvi_mosaic, ndvi_transform = merge(src_files)

# Copy metadata
meta = src_files[0].meta.copy()

meta.update({
    "driver": "GTiff",
    "height": ndvi_mosaic.shape[1],
    "width": ndvi_mosaic.shape[2],
    "transform": ndvi_transform,
    "count": 1
})

# Output path
output_ndvi = PROCESSED_DIR / "ndvi.tif"

# Save merged raster
with rasterio.open(output_ndvi, "w", **meta) as dst:
    dst.write(ndvi_mosaic)

# Close files
for src in src_files:
    src.close()

print("\n" + "=" * 70)
print("✅ NDVI merged successfully!")
print("=" * 70)
print("Saved to:", output_ndvi)

NDVI Files Found
ner_ndvi.tif
ner_ndvi2.tif

✅ NDVI merged successfully!
Saved to: /Users/rajdeepmandal/Desktop/PrithviX/Processed/ndvi.tif


In [ ]:
# ==========================================================
# Cell 5 - Verify Merged NDVI
# ==========================================================

import rasterio
import matplotlib.pyplot as plt

with rasterio.open(PROCESSED_DIR / "ndvi.tif") as src:

    print("=" * 60)
    print("Merged NDVI Information")
    print("=" * 60)

    print("Width      :", src.width)
    print("Height     :", src.height)
    print("CRS        :", src.crs)
    print("Bands      :", src.count)
    print("Resolution :", src.res)

    ndvi = src.read(1)

plt.figure(figsize=(10,8))
plt.imshow(ndvi, cmap="RdYlGn")
plt.colorbar(label="NDVI")
plt.title("Merged NDVI")
plt.axis("off")
plt.show()

Merged NDVI Information
Width      : 38962
Height     : 27475
CRS        : EPSG:4326
Bands      : 1
Resolution : (0.00026949458523585647, 0.00026949458523585647)


In [ ]:
import shutil

# DEM
shutil.copy(DEM_FILE, PROCESSED_DIR / "dem.tif")

# Soil
shutil.copy(SOIL_FILE, PROCESSED_DIR / "soil.tif")

# Population
shutil.copy(POPULATION_FILE, PROCESSED_DIR / "population.tif")

print("Datasets copied successfully.")

In [1]:
import platform
import psutil

print(platform.platform())
print("RAM:", round(psutil.virtual_memory().total / (1024**3), 2), "GB")

macOS-15.3-arm64-arm-64bit-Mach-O
RAM: 16.0 GB
